In [2]:
from kloppy import statsbomb

# Load a specific match from open data
dataset = statsbomb.load_open_data(
    match_id=3788741,  # Example: Barcelona vs Real Madrid
    coordinates="statsbomb"  # Coordinate system
)

# View basic info
print(f"Home team: {dataset.metadata.teams[0].name}")
print(f"Away team: {dataset.metadata.teams[1].name}")
print(f"Score: {dataset.metadata.score}")
print(f"Total events: {len(dataset.events)}")

# Convert to pandas DataFrame
df = dataset.to_df()
print(df.head())

d:\SASUniversityEdition\Machine\MODEL\football\.conda\Lib\site-packages\kloppy\_providers\statsbomb.py:83: UserWarning: 

You are about to use StatsBomb public data.
By using this data, you are agreeing to the user agreement. 
The user agreement can be found here: https://github.com/statsbomb/open-data/blob/master/LICENSE.pdf

  warnings.warn(


Home team: Turkey
Away team: Italy
Score: None
Total events: 3911
                               event_id           event_type  period_id  \
0  59645e89-9c7f-46f2-b662-332c5b8bba12  GENERIC:Starting XI          1   
1  f063a4b6-f5ad-4576-b31d-a8dbe50f693e  GENERIC:Starting XI          1   
2  4d9693c5-29bb-4e73-8913-7ee7a4e465fc   GENERIC:Half Start          1   
3  5c53ec59-b0d0-4515-8fee-d973e0037b56   GENERIC:Half Start          1   
4  5c888f58-fe77-459b-ab3b-a2fa5fb8ab16                 PASS          1   

               timestamp          end_timestamp ball_state ball_owning_team  \
0        0 days 00:00:00                    NaT      alive              909   
1        0 days 00:00:00                    NaT      alive              909   
2        0 days 00:00:00                    NaT      alive              909   
3        0 days 00:00:00                    NaT      alive              909   
4 0 days 00:00:00.878000 0 days 00:00:02.788504      alive              909   

  team_i

In [8]:
import kloppy
from kloppy import statsbomb
from kloppy.domain import EventType, PassResult, ShotResult # Import ShotResult

# Load a sample dataset from StatsBomb's open data
dataset = statsbomb.load_open_data(
    match_id=15946,  # You can change this ID for different matches
    coordinates="statsbomb"
)

# Get the list of events
events = dataset.events

# --- Calculate Pass-Related Statistics ---
all_passes = [event for event in events if event.event_type == EventType.PASS]
completed_passes = [pass_ for pass_ in all_passes if pass_.result and pass_.result.COMPLETE]

total_passes = len(all_passes)
pass_completion_rate = len(completed_passes) / total_passes if total_passes > 0 else 0

print(f"Total Passes: {total_passes}")
print(f"Pass Completion Rate: {pass_completion_rate:.2%}")

## --- Calculate Shots and Shots on Target ---
all_shots = [event for event in events if event.event_type == EventType.SHOT]

# Shots on target include: GOAL, SAVED, and POST (since hitting post is on target)
shots_on_target = [shot for shot in all_shots if shot.result in [
    ShotResult.GOAL, 
    ShotResult.SAVED, 
    ShotResult.POST
]]

# Shots off target include: OFF_TARGET, BLOCKED
shots_off_target = [shot for shot in all_shots if shot.result in [
    ShotResult.OFF_TARGET, 
    ShotResult.BLOCKED
]]

# OWN_GOAL is a special case - you might want to track it separately
own_goals = [shot for shot in all_shots if shot.result == ShotResult.OWN_GOAL]

shot_accuracy = len(shots_on_target) / len(all_shots) if len(all_shots) > 0 else 0

print(f"Total Shots: {len(all_shots)}")
print(f"Shots on Target: {len(shots_on_target)}")
print(f"Shots off Target: {len(shots_off_target)}")
print(f"Own Goals: {len(own_goals)}")
print(f"Shot Accuracy: {shot_accuracy:.2%}")
# Note: You can extend this logic to filter passes by pitch area (defensive, middle, attacking third)
# by checking the coordinates of the pass event (event.coordinates.x and event.coordinates.y)

Total Passes: 1163
Pass Completion Rate: 99.66%
Total Shots: 28
Shots on Target: 11
Shots off Target: 17
Own Goals: 0
Shot Accuracy: 39.29%


In [9]:
from kloppy.domain import EventType, PassResult, ShotResult
import pandas as pd

def analyze_shots(events):
    """Comprehensive shot analysis including xG"""
    all_shots = [event for event in events if event.event_type == EventType.SHOT]
    
    shot_data = []
    for shot in all_shots:
        shot_info = {
            'team': shot.team.name if shot.team else 'Unknown',
            'player': shot.player.name if shot.player else 'Unknown',
            'result': shot.result.value,
            'timestamp': shot.timestamp,
            'coordinates': shot.coordinates
        }
        
        # Add xG if available (from custom StatsbombShotEvent)
        if hasattr(shot, 'xg'):
            shot_info['xg'] = shot.xg
        elif hasattr(shot, 'result') and shot.result == ShotResult.GOAL:
            shot_info['xg'] = 1.0  # Actual goal
        else:
            shot_info['xg'] = 0.0
            
        shot_data.append(shot_info)
    
    # Convert to DataFrame for easier analysis
    shots_df = pd.DataFrame(shot_data)
    
    if not shots_df.empty:
        # Calculate summary statistics
        total_shots = len(shots_df)
        shots_on_target = len(shots_df[shots_df['result'].isin(['GOAL', 'SAVED', 'POST'])])
        total_goals = len(shots_df[shots_df['result'] == 'GOAL'])
        total_xg = shots_df['xg'].sum()
        
        print(f"Total Shots: {total_shots}")
        print(f"Shots on Target: {shots_on_target}")
        print(f"Goals: {total_goals}")
        print(f"Total xG: {total_xg:.2f}")
        print(f"Shot Accuracy: {shots_on_target/total_shots:.2%}" if total_shots > 0 else "Shot Accuracy: 0%")
        print(f"Conversion Rate: {total_goals/total_shots:.2%}" if total_shots > 0 else "Conversion Rate: 0%")
        
        # Team-level analysis
        team_stats = shots_df.groupby('team').agg({
            'xg': 'sum',
            'result': 'count'
        }).rename(columns={'result': 'shots'})
        
        team_stats['xg_per_shot'] = team_stats['xg'] / team_stats['shots']
        print("\nTeam Shot Statistics:")
        print(team_stats)
    
    return shots_df

# Usage
shots_df = analyze_shots(events)

Total Shots: 28
Shots on Target: 11
Goals: 3
Total xG: 3.00
Shot Accuracy: 39.29%
Conversion Rate: 10.71%

Team Shot Statistics:
                   xg  shots  xg_per_shot
team                                     
Barcelona         3.0     25         0.12
Deportivo Alavés  0.0      3         0.00


In [ ]:
from kloppy import statsbomb
from kloppy.domain import ShotEvent, EventFactory, create_event
from dataclasses import dataclass
from typing import Optional

# 1. Define a custom shot event class to include xG
@dataclass(repr=False)
class StatsbombShotEvent(ShotEvent):
    xg: Optional[float] = None

    @property
    def inverted_xg(self) -> Optional[float]:
        if self.xg is not None:
            return 1 - self.xg
        else:
            return None

    def __str__(self):
        return f"<Shot player='{self.player}' xg={self.xg} result='{self.result}'>"

# 2. Implement a custom factory to extract xG
class StatsbombEventFactory(EventFactory):
    def build_shot(self, raw_event, **kwargs) -> ShotEvent:
        xg = (
            raw_event["shot"]["statsbomb_xg"]
            if "shot" in raw_event and raw_event["shot"]
            else None
        )
        if xg is not None:
            xg = float(xg)

        return create_event(
            StatsbombShotEvent,
            xg=xg,
            raw_event=raw_event,
            **kwargs,
        )

# 3. Load data with the custom factory to get xG for each shot
dataset = statsbomb.load_open_data(
    match_id=15946,
    coordinates="statsbomb",
    event_factory=StatsbombEventFactory()  # Use our custom factory
)

shots = dataset.filter("shot")
for shot in shots[:5]:  # Inspect the first 5 shots
    print(shot)

# You can then aggregate xG by team for a match.

<Shot player='Lionel Andrés Messi Cuccittini' xg=0.07699243 result='OFF_TARGET'>
<Shot player='Jordi Alba Ramos' xg=0.05166811 result='OFF_TARGET'>
<Shot player='Lionel Andrés Messi Cuccittini' xg=0.016932096 result='SAVED'>
<Shot player='Rubén Sobrino Pozuelo' xg=0.1226044 result='OFF_TARGET'>
<Shot player='Luis Alberto Suárez Díaz' xg=0.041750744 result='OFF_TARGET'>
